# Wine Quality Regression

Predict the wine `quality` score from the other white wine variables.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/YOUR_REPOSITORY_NAME/blob/main/wine_quality_regression.ipynb)


## Imports
These libraries are used to load the data, build the model, train it, and plot the results.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
TEST_SIZE = 0.20
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 0.001

np.random.seed(SEED)
torch.manual_seed(SEED)


## Load The Data
This cell loads the white wine dataset with 4,898 examples.


In [ ]:
data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"
local_path = Path("winequality-white.csv")

if local_path.exists():
    df = pd.read_csv(local_path, sep=";")
else:
    df = pd.read_csv(data_url, sep=";")

print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")

X = df.drop(columns=["quality"]).values.astype(np.float32)
y = df["quality"].values.astype(np.float32).reshape(-1, 1)

print(f"Input shape: {X.shape}")
print(f"Output shape: {y.shape}")


## Train And Test Datasets
This cell creates the train dataset and the independent test dataset, and scales the input values.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Train dataset size: {len(X_train)}")
print(f"Independent test dataset size: {len(X_test)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Batches per epoch: {len(train_loader)}")


## Model And Loss Function
This cell defines the regression model, the loss function, and the optimizer.


In [ ]:
class Net(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.layers(x)

model = Net(X_train.shape[1])
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)
print(f"Loss function: {loss_fn.__class__.__name__}")
print(f"Epochs: {EPOCHS}")


## Training
This cell trains the model for several epochs using batches from the train dataset.


In [ ]:
loss_history = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for batch_inputs, batch_targets in train_loader:
        predictions = model(batch_inputs)
        loss = loss_fn(predictions, batch_targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    loss_history.append(epoch_loss)

    if epoch == 0 or (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1:3d}/{EPOCHS} - loss: {epoch_loss:.4f}")


## Prediction And Evaluation
This cell uses the trained model to predict the quality values on the independent test dataset.


In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(torch.tensor(X_test, dtype=torch.float32)).numpy().flatten()

y_true = y_test.flatten()

mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R^2:  {r2:.4f}")


## Plot 1
This plot compares the actual quality values with the predicted quality values.


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_true, y_pred, alpha=0.45)
line_start = min(y_true.min(), y_pred.min())
line_end = max(y_true.max(), y_pred.max())
plt.plot([line_start, line_end], [line_start, line_end], "r--")
plt.xlabel("Actual quality")
plt.ylabel("Predicted quality")
plt.title("Actual vs Predicted")
plt.grid(alpha=0.3)
plt.show()


## Plot 2
This plot shows the training loss versus the number of epochs.


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, EPOCHS + 1), loss_history)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss vs Epochs")
plt.grid(alpha=0.3)
plt.show()


## Assignment Keywords
- input: `X`
- output: `y`
- model: `Net`
- loss function: `loss_fn`
- epoch: `for epoch in range(EPOCHS)`
- batch: `train_loader` with batch size 64
- predict: `y_pred` on the test set
- train dataset: `X_train`, `y_train`
- independent test dataset: `X_test`, `y_test`
